# 🧠🤖 第6周-Day4：推理能力评测与Prompt工程

> 📚 学习目标：掌握大模型推理能力的评测方法，理解Prompt工程的核心策略

---

## 📖 今日概览

昨天我们学了 **ReAct模式与多Agent协作**，理解了Agent如何在思考和行动之间循环推进。

今天我们进入 **推理能力评测** 的世界——
- 🎯 大模型推理到底怎么评测？
- 🔧 Prompt工程的核心技巧有哪些？
- 📊 零样本 vs 少样本，哪个更有效？
- ✅ 自洽性检查如何提升准确率？

## 🔄 昨日复习

**ReAct模式与多Agent协作**
- ReAct = Reasoning + Acting，推理与行动交替推进
- 思考→行动→观察→继续推理，形成闭环
- 多Agent协作：每个Agent专注特定任务，主Agent统筹全局
- 关键：工具选择策略类似于Transformer中的注意力机制

## 📝 核心知识点

### 1️⃣ 大模型推理评测三大基准

| 基准 | 全称 | 评测维度 | 题目类型 |
|------|------|----------|----------|
| **MMLU** | Massive Multitask Language Understanding | 知识广度 | 57个学科选择题 |
| **GSM8K** | Grade School Math 8K | 数学推理 | 小学数学应用题 |
| **HumanEval** | HumanEval | 代码生成 | 函数补全任务 |

- **MMLU** 测试模型的"知识面"，从法律到医学到计算机
- **GSM8K** 专注"数学推理"，需要多步计算
- **HumanEval** 衡量"代码能力"，给定函数签名和文档，补全实现

### 2️⃣ Prompt工程核心策略

**零样本（Zero-shot）**：直接提问，不给任何示例
```
❌ "法国首都？" → 可能答错
```

**少样本（Few-shot）**：给1-2个示例再提问
```
✅ Q: 中国首都？ A: 北京
   Q: 日本首都？ A: 东京
   Q: 法国首都？ A: 巴黎
```

**思维链（Chain-of-Thought）**：要求展示推理过程
```
✅ "请一步步思考：25的20%是多少？"
   → 25 × 0.20 = 5
```

**自洽性（Self-consistency）**：多次采样，选最一致的答案
- 生成5-10个推理路径
- 如果7次都答5，1次答4 → 答案是5
- 本质是"少数服从多数"，但只对推理类问题有效

In [ ]:
# 配置 matplotlib 中文显示
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
import numpy as np

# 清除 matplotlib 字体缓存
cache_dir = matplotlib.get_cachedir()
for item in os.listdir(cache_dir):
    if item.startswith('fontlist'):
        os.remove(os.path.join(cache_dir, item))

# 重新构建字体列表
fm._load_fontmanager(try_read_cache=False)

# 配置中文字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK JP', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('✅ 中文字体配置完成')

In [ ]:
# 不同Prompt策略在各类任务上的准确率对比
strategies = ['零样本', '少样本', '思维链', '自洽性']
math_acc = [45, 62, 78, 85]      # 数学推理
logic_acc = [40, 55, 72, 80]     # 逻辑推理
fact_acc = [70, 73, 71, 72]      # 事实问答（推理策略帮助不大）

x = np.arange(len(strategies))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, math_acc, width, label='数学推理', color='#FF6B6B', alpha=0.85)
bars2 = ax.bar(x, logic_acc, width, label='逻辑推理', color='#4ECDC4', alpha=0.85)
bars3 = ax.bar(x + width, fact_acc, width, label='事实问答', color='#45B7D1', alpha=0.85)

ax.set_ylabel('准确率 (%)', fontsize=14)
ax.set_title('不同Prompt策略的推理准确率对比', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(strategies, fontsize=13)
ax.legend(fontsize=12)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

# 添加数值标签
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第6周/prompt_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 图表已保存')

### 3️⃣ Prompt工程的RTF框架

**R - Role（角色设定）**：告诉模型它应该扮演什么角色
```
"你是一位资深AI工程师，精通大模型推理技术..."
```

**T - Task（任务描述）**：清晰定义你要它做什么
```
"请分析以下数学问题的推理过程，指出其中的错误..."
```

**F - Format（输出格式）**：规定回答的结构和形式
```
"请用以下格式回答：
1. 问题分析
2. 推理步骤
3. 最终答案"
```

💡 **RTF框架** 是Prompt工程的基础方法论，几乎所有高质量Prompt都遵循这个结构。

In [ ]:
# 模拟自洽性检查（Self-consistency）
import random
from collections import Counter

def self_consistency_check(problem, num_samples=10):
    """
    模拟自洽性检查流程
    - 生成多个推理路径
    - 统计答案分布
    - 选择最一致的答案
    """
    print(f'📝 问题: {problem}')
    print(f'🔬 生成 {num_samples} 条推理路径...\n')
    
    # 模拟模型生成多个答案（真实场景中是多次LLM调用）
    # 假设正确答案是42，模型在多次推理中大部分能得出正确答案
    correct_answer = 42
    answers = []
    for i in range(num_samples):
        # 70%概率答对，30%概率答错（模拟真实模型表现）
        if random.random() < 0.70:
            answer = correct_answer
            status = '✅'
        else:
            answer = random.choice([40, 41, 43, 44])
            status = '❌'
        answers.append(answer)
        print(f'  路径{i+1}: 推理... → 答案={answer} {status}')
    
    # 统计最频繁答案
    counter = Counter(answers)
    final_answer = counter.most_common(1)[0][0]
    confidence = counter.most_common(1)[0][1] / num_samples * 100
    
    print(f'\n📊 答案分布: {dict(counter)}')
    print(f'🎯 最终答案: {final_answer} (置信度: {confidence:.0f}%)')
    print(f'{'✅ 正确！' if final_answer == correct_answer else '❌ 错误'}')
    return final_answer

self_consistency_check('小明有25个苹果，卖掉了20%，还剩多少个？')

In [ ]:
# 大模型评测维度雷达图
categories = ['知识理解', '数学推理', '代码生成', '逻辑分析', '创意生成', '安全性']
N = len(categories)

# 模拟不同模型的评测分数
model_a = [85, 72, 78, 70, 82, 90]  # GPT-4级别
model_b = [80, 88, 85, 75, 75, 85]  # DeepSeek级别
model_c = [70, 65, 60, 68, 78, 92]  # 对齐优化模型

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for model, scores, color, label in [
    ('模型A', model_a, '#FF6B6B', 'GPT-4级别'),
    ('模型B', model_b, '#4ECDC4', 'DeepSeek级别'),
    ('模型C', model_c, '#45B7D1', '对齐优化模型')
]:
    values = scores + scores[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=label, color=color, markersize=6)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=13)
ax.set_ylim(0, 100)
ax.set_title('大模型推理能力评测雷达图', fontsize=16, fontweight='bold', y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第6周/model_evaluation_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 雷达图已保存')

## ✏️ 课堂练习（5分钟）

❶ MMLU基准主要评测大模型的什么能力？

❷ 零样本和少样本学习的区别是什么？

❸ 自洽性检查为什么重要？

💡 提示：从评测目的和适用场景思考

## 📝 课后测试（15分钟）

❶ MMLU基准主要评测大模型的什么能力？
   A. 数学推理能力
   B. 代码生成能力  
   C. 知识理解和综合应用能力
   D. 创意写作能力

❷ 以下哪种prompt技术最适合解决复杂的多步推理问题？
   A. 零样本直接提问
   B. Chain-of-Thought
   C. 简单指令
   D. 仅提供答案

❸ 什么是prompt工程的RTF框架？
   A. Role-Task-Format
   B. Reasoning-Training-Format
   C. Response-Task-Format
   D. Role-Technology-Format

❹ 自洽性检查的目的是什么？
   A. 让模型回答更快
   B. 提高回答准确性
   C. 减少模型计算量
   D. 增加回答长度

❺ 评测大模型推理能力时，为什么需要多个基准测试？
   A. 单个基准已经足够
   B. 不同基准测试不同认知维度
   C. 为了让模型通过所有测试
   D. 为了节省评测时间

回复答案我帮你批改 ✅

## 📌 今日总结

| 概念 | 一句话 |
|------|--------|
| MMLU | 知识广度基准，57个学科 |
| GSM8K | 数学推理基准，多步计算 |
| HumanEval | 代码生成基准，函数补全 |
| 零样本 | 直接提问，不给示例 |
| 少样本 | 给1-2个示例再提问 |
| CoT | 要求展示推理过程 |
| 自洽性 | 多次采样，选最一致答案 |
| RTF框架 | Role + Task + Format |

🔄 **往期回顾**：Transformer中的自注意力机制如何影响推理能力？

💡 **进度**：第6周/12 | 当前：推理与思维链

🚨 明天 Day5 将学习 **Agent开发实战框架设计**，把本周的推理能力整合进Agent架构中！

## 🔑 今日英文术语

| 术语 | 音标 | 中文释义 |
|------|------|----------|
| **MMLU** | /em em el juː/ | 大规模多任务理解基准 |
| **GSM8K** | /dʒi es em eɪt keɪ/ | 小学数学8000题评测 |
| **HumanEval** | /ˈhjuːmənɪˈvæljueɪt/ | 人类代码评测基准 |
| **Zero-shot** | /ˈzɪərəʊ ʃɒt/ | 零样本 |
| **Few-shot** | /fjuː ʃɒt/ | 少样本 |
| **Chain-of-Thought** | /ʃeɪn əv θɔːt/ | 思维链 |
| **Self-consistency** | /self kənˈsɪstənsi/ | 自洽性 |
| **Prompt Engineering** | /prɒmpt ˈendʒɪˈnɪərɪŋ/ | 提示工程 |
| **Ground Truth** | /ɡraʊnd truːθ/ | 真实答案 |
| **Hallucination** | /həˌluːsɪˈneɪʃən/ | 幻觉 |